# 公式チュートリアル 02 — How to Build a Model
> 出典: https://ecell4.e-cell.org/tutorials/tutorial02.html

同じモデルを**4 通り**の書き方で作れる: (1) DSL `reaction_rules()`、(2) `ReactionRule` オブジェクト、
(3) `create_*_reaction_rule` ファクトリ、(4) デコレータ。

In [1]:
import warnings; warnings.filterwarnings('ignore')
from ecell4.prelude import *

# (1) DSL
with reaction_rules():
    A + B == C | (0.01, 0.3)
m_dsl = get_model()

# (2) ReactionRule オブジェクト
m_obj = NetworkModel()
m_obj.add_reaction_rule(ReactionRule([Species('A'), Species('B')], [Species('C')], 0.01))
m_obj.add_reaction_rule(ReactionRule([Species('C')], [Species('A'), Species('B')], 0.3))

# (3) ファクトリ関数
m_fac = NetworkModel()
m_fac.add_reaction_rule(create_binding_reaction_rule(Species('A'), Species('B'), Species('C'), 0.01))
m_fac.add_reaction_rule(create_unbinding_reaction_rule(Species('C'), Species('A'), Species('B'), 0.3))

# (4) デコレータ
@reaction_rules
def rrgen(kon, koff):
    A + B == C | (kon, koff)
m_dec = NetworkModel(); m_dec.add_reaction_rules(rrgen(0.01, 0.3))

for name, mm in [('DSL', m_dsl), ('objects', m_obj), ('factory', m_fac), ('decorator', m_dec)]:
    print(f'{name:10s}: {len(mm.reaction_rules())} reaction rules, species {[s.serial() for s in mm.list_species()]}')

DSL       : 2 reaction rules, species ['A', 'B', 'C']
objects   : 2 reaction rules, species ['A', 'B', 'C']
factory   : 2 reaction rules, species ['A', 'B', 'C']
decorator : 2 reaction rules, species ['A', 'B', 'C']


**要点**: どの書き方でも同じ `A + B ⇌ C`（2 反応・3 種）になる。合成/分解は `~A > A` / `A > ~A`、
連結記法 `(E + S == ES | (kf,kr) > E + P | kcat)` で酵素反応も書ける（tut06 参照）。掃引で作り直すなら
オブジェクト/ファクトリ（グローバル累積を避ける）が安全。